In [0]:
from datetime import datetime

def log_silver(run_id, resource, status, start_time, count, message):

    end_time = datetime.now()

    data = [(run_id, resource, status, start_time, end_time, count, message)]

    df = spark.createDataFrame(
        data,
        ["run_id","resource","status","start_time","end_time","record_count","message"]
    )

    df.write.mode("append").saveAsTable("fhir_catalog.audit.silver_log")

In [0]:
from pyspark.sql.functions import col, coalesce, expr, to_date, max as spark_max
from datetime import datetime
import uuid

def silver_transform_incremental(resource):

    run_id = str(uuid.uuid4())
    start_time = datetime.now()

    bronze_path = f"/Volumes/fhir_catalog/bronze/fhir_bronze_vol/{resource.lower()}/parquet/"
    silver_table = f"fhir_catalog.silver.{resource.lower()}"
    watermark_table = "fhir_catalog.silver.watermark_tracker"

    try:
        df = spark.read.parquet(bronze_path)
        wm_df = spark.sql(f"""
            SELECT last_processed_ts 
            FROM {watermark_table}
            WHERE resource = '{resource}'
        """)

        if wm_df.count() > 0:
            last_wm = wm_df.collect()[0][0]
            df = df.filter(col("extraction_timestamp") >= last_wm)

        if resource.lower() == "patient":
            final_df = df.select(
                col("id").alias("patient_id"),
                col("resource_data.gender").alias("gender"),
                col("resource_data.birthDate").alias("birth_date"),
                col("resource_data.meta.lastUpdated").alias("last_updated"),
                to_date(col("extraction_timestamp")).alias("extract_date")
            )
            key = "patient_id"

        elif resource.lower() == "encounter":
            final_df = df.select(
                col("id").alias("encounter_id"),
                col("resource_data.status").alias("status"),
                col("resource_data.subject.reference").alias("patient_ref"),
                col("resource_data.period.start").alias("start_time"),
                to_date(col("extraction_timestamp")).alias("extract_date")
            )
            key = "encounter_id"

        elif resource.lower() == "observation":
            final_df = df.select(
                col("id").alias("observation_id"),
                col("resource_data.status").alias("status"),
                coalesce(
                    col("resource_data.code.text"),
                    expr("resource_data.code.coding[0].code")
                ).alias("observation_type"),
                col("resource_data.valueQuantity.value").alias("value"),
                col("resource_data.subject.reference").alias("patient_ref"),
                to_date(col("extraction_timestamp")).alias("extract_date")
            )
            key = "observation_id"
            
            
        elif resource.lower() == "condition":
            final_df = df.select(
            col("id").alias("condition_id"),
            expr("resource_data.clinicalStatus.coding[0].code").alias("clinical_status"),
            coalesce(
            expr("resource_data.code.coding[0].code"),
            expr("resource_data.code.text")).alias("condition"),

            col("resource_data.onsetDateTime").alias("onset_date"),
            col("resource_data.meta.lastUpdated").alias("last_updated"),
            to_date(col("extraction_timestamp")).alias("extract_date"))

            key = "condition_id"

        else:
            raise ValueError("Unsupported resource")

# Deduplication and null check
        final_df = final_df.dropDuplicates([key])
        final_df = final_df.fillna("UNKNOWN")

        count = final_df.count()

        if count == 0:
            log_silver(run_id, resource, "SUCCESS", start_time, 0, "No new data")
            print(f"{resource}: no new data")
            return

        final_df.write.format("delta") \
            .mode("append") \
            .saveAsTable(silver_table)

        new_wm = df.agg(spark_max("extraction_timestamp")).collect()[0][0]

        if new_wm:
            spark.sql(f"""
                MERGE INTO {watermark_table} t
                USING (SELECT '{resource}' AS resource, '{new_wm}' AS last_processed_ts) s
                ON t.resource = s.resource
                WHEN MATCHED THEN UPDATE SET last_processed_ts = s.last_processed_ts
                WHEN NOT MATCHED THEN INSERT *
            """)

        log_silver(run_id, resource, "SUCCESS", start_time, count, "Processed")

        print(f"{resource} silver completed")

    except Exception as e:

        log_silver(run_id, resource, "FAILED", start_time, 0, str(e))

        print(f"{resource} FAILED")
        raise

In [0]:
resources = ["Patient", "Encounter", "Observation", "Condition"]

for r in resources:
    silver_transform_incremental(r)